In [ ]:
# Set run type - Allows user to run code in DEMO state (using dummy dataset) or Full Version, accessing resources from Google Colab
# Set 'ON' to run Demo version, 'OFF' to run Full Version (need access to Google Colab resources)
import os
os.environ['RUN_DEMO'] = 'ON' 

## Libraries and Imports

In [ ]:
import src.config as c
import src.load_data as ld
viirs_files = ld.get_filepaths('VIIRS')
viirs_to_load = ld.to_load_viirs(viirs_files,[2018])
viirs_to_load

In [ ]:
import src.load_data as ld

In [ ]:
# import sys
# import os

# # Set working directory and mount GoogleDrive
# drive.mount('/content/drive')
# wd = '/content/drive/MyDrive/Colab Notebooks/WildFirePrediction/[2]ProgramWildFirePredict'
# if os.path.exists(wd):
#   print(f"✅  Success - working directory (wd) set as [{wd}]")
# else:
#   print(f"❌ Error: Could not set [{wd}] as root directory for project. Review path and try again...")
# sys.path.append(wd)

In [ ]:
import pandas as pd

d1 = {'lon':     [1,            2,             3,           4,            5],
      'lat':     [10,           11             ,12,         13,           14],
      'date':    ['2024-01-01', '2024-02-02', '2024-03-03', '2024-04-04', '2024-05-05'],
      'another': [0.1,          0.2,           0.3,         0.4,           0.5]}

d2 = {'lon':     [1,            3,            9,            4,            5],
      'lat':     [10,           11,           12,           20,           14],
      'date':    ['2024-01-01', '2024-02-02', '2025-03-03', '2024-04-04', '2024-05-05'],
      'another': [0.1,          0.2,           0.3,          0.4,          0.101]}
                 
de = {'lon':     [1,            2,             3,           4,            5,             3,           9,            4          ],
      'lat':     [10,           11             ,12,         13,           14,            11,          12,           20         ],
      'date':    ['2024-01-01', '2024-02-02', '2024-03-03', '2024-04-04', '2024-05-05',  '2024-02-02','2025-03-03', '2024-04-04'],
      'another': [0.1,          0.2,           0.3,         0.4,           0.5,          0.2,          0.3,          0.4       ]}

cols_merge = ['lon','lat','date']
df1 = pd.DataFrame(d1)
df2 = pd.DataFrame(d2)

df_diff = df2.loc[ ~df2.set_index(cols_merge).index.isin(df1.set_index(cols_merge).index)]
df_diff

df_merged = pd.concat([df1,df_diff], ignore_index=True)
df_merged.shape[0]


In [1]:
import os
os.environ.setdefault("RUN_DEMO", "ON")
import src.config as c
import src.load_data as ld
import geopandas as gpd
from pathlib import Path
import matplotlib.pyplot as plt
import utils as u

# --------------------------
# VARIABLES
# --------------------------
YEAR_FILTER = [2019]
CRS = "EPSG: 4326"          # Set Coordinate Reference System (CRS) so it is uniform across all data inputs
SATELITE_IMAGES = "COPERNICUS/S2_SR_HARMONIZED"         
# --------------------------
# VIIRS DATA
# --------------------------
viirs_dict = ld.viirs_load_pipeline(dir_name = 'VIIRS',
                                    crs = CRS,
                                    date_range = YEAR_FILTER)
df_viirs = viirs_dict.get('df_viirs')
print(f"{'='*80}")
print(f"VIIRS Data")
print(f"\tData Type: {type(df_viirs)}")
print(f"\t📅 Date Range: {df_viirs['acq_date'].min()} to {df_viirs['acq_date'].max()}")


# --------------------------
# UK GRID 
# --------------------------
df_uk_grid = ld.load_uk_grid(file_name='ukcp18-uk-land-12km.shp',
                             crs=CRS)
print(f"{'='*80}")
print(f"UK Grid")
print(f"Shape: {df_uk_grid.shape}")

# Grids by Day
print(f"{'='*80}")
print(f"UK Grid Daily")
dates = u.extract_year_range(df_viirs)
df_daily_grid = df_uk_grid.copy()
df_daily_grid['join_key'] = 1
df_daily_grid = df_daily_grid.merge(dates, on='join_key').drop(columns='join_key')
print(f"Shape: {df_daily_grid.shape}")
print(f"Columns: {df_daily_grid.columns}")



VIIRS Data
	Data Type: <class 'geopandas.geodataframe.GeoDataFrame'>
	📅 Date Range: 2019-01-01 to 2019-12-31
UK Grid
Shape: (1692, 4)
UK Grid Daily
Shape: (617580, 5)
Columns: Index(['id', 'x_coord', 'y_coord', 'geometry', 'date'], dtype='object')


In [33]:
uk_grid_sample = df_daily_grid[df_daily_grid['date'] == '2019-01-01']
#uk_grid_sample.explore()
uk_grid_sample.head()

,id,x_coord,y_coord,geometry,date
0,1,174000.0,18000.0,"POLYGON ((-5.23625 49.963, -5.2435 50.07075, -...",2019-01-01
365,2,150000.0,30000.0,"POLYGON ((-5.57824 50.0609, -5.58629 50.16861,...",2019-01-01
730,3,162000.0,30000.0,"POLYGON ((-5.41089 50.06595, -5.41857 50.17368...",2019-01-01
1095,4,174000.0,30000.0,"POLYGON ((-5.2435 50.07075, -5.2508 50.1785, -...",2019-01-01
1460,5,162000.0,42000.0,"POLYGON ((-5.41857 50.17368, -5.42629 50.28141...",2019-01-01


In [3]:
import ee
SATELITE_IMAGES = "COPERNICUS/S2_SR_HARMONIZED"         

try: 
    ee.Initialize(project="ee-enmanuelmorego")
except:
    ee.Authenticate
    ee.Initialize(project="ee-enmanuelmorego")

s2 = ee.ImageCollection(SATELITE_IMAGES)

In [ ]:
uk_grid_sample

In [34]:
import pandas as pd
def geodf_to_ee(geo_df: gpd.GeoDataFrame) -> ee.FeatureCollection:
    """
    Convert GeoPandasDataFrame to a GoogleEarth Feature Collection to extract Sentinel-2 images

    Args:
        geo_df (GeoDataFrame): Data frame containing the plygon for the UK Grids, for each date of the date range

    Returns:
        Feature Collection
    """
    features = []
    for _, r in geo_df.iterrows():
        date_f  = r["date"].strftime("%Y-%m-%d")
        coords  = [list(r.geometry.exterior.coords)]
        geom    = ee.Geometry.Polygon(coords)
        feature = ee.Feature(geom,
                             {'date'   : date_f,
                              'grid_id': r['id']})
        features.append(feature)
    return ee.FeatureCollection(features)

def attach_s2_metadata(feature: ee.Feature) -> ee.Feature:
    """
    Takes the Feature of a Feature Collection object constructed from the UK Grid data, and returns
    the metadata for the matching images in Sentinel-2 (sentinel-2 is a global ee variable, defined outside this function)
    - Read FC date
    - Read FC Geometry
    - Find matching images in Sentinel-2
        - Date + 1 day window
        - Overlaps with geometry (Defined UK Grids)
    - Choose one image (least cloudy, better quality, more information)
    - Attach metadata to feature
    """
    # Extract date
    date = ee.Date(feature.get("date"))
    # Extract Geometry
    geom = feature.geometry()
    # Filter Sentinel Class by date and Geometry (location)
    s2_day = (s2
              .filterDate(date, date.advance(1,"day"))
              .filterBounds(geom)
              .sort("CLOUDY_PIXEL_PERCENTAGE"))
    # Choose least cloudy image
    img = s2_day.first()
    # Return results
    return feature.set({"s2_id": ee.Algorithms.If(img, img.get("system:id"), None),
                        "cloud_pct": ee.Algorithms.If(img, img.get("CLOUDY_PIXEL_PERCENTAGE"), None),
                        "geometry": geom
                        })

def s2_metadata_to_df(fc: ee.FeatureCollection) -> pd.DataFrame:
    """
    Function that takes the feature collection containing the Sentinel-2 Metadata and
    transforms it into a pandas DataFrame

    Note:
        If no data is found for a given grid_id + date, this will be populated with `None` and dealt
        with at a later stage

    Args:
        fc (Feature collection): Contained S2 metadata

    Returns:
        df (dataframe): {date, grid_id, sentinel_id, cloud_pct}
    """
    fc_info = fc.getInfo()
    rows = []
    for f in fc_info['features']:
        properties          = f['properties']
        row = {"date"       : properties["date"],
               "grid_id"    : properties["grid_id"],
               "sentinel_id": properties.get("s2_id"),
               "cloud_pct"  : properties.get("cloud_pct")}
        rows.append(row)
    df_out = pd.DataFrame(rows)
    return df_out


In [35]:
fc_uk_grid = geodf_to_ee(uk_grid_sample)
# Get Sentinel-2 Archive (this is set as a Global ee Variable, therefore, no need to pass in function below - it'll be visible to the function)
s2 = ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
fc_w_s2 = fc_uk_grid.map(attach_s2_metadata)
# Convert S2 Metadata info a DF with Sentinel-2 Metadata attached (for each day-grid combination)
df_s2 = s2_metadata_to_df(fc_w_s2)



In [36]:
df_s2.head()

,date,grid_id,sentinel_id,cloud_pct
0,2019-01-01,1,None,NaN
1,2019-01-01,2,None,NaN
2,2019-01-01,3,None,NaN
3,2019-01-01,4,None,NaN
4,2019-01-01,5,None,NaN


In [37]:
df_s2_pop = df_s2[df_s2.sentinel_id.notnull()]
print(f"Original Shape {df_s2.shape}")
print(f"Cleaned Shape  {df_s2_pop.shape}")

Original Shape (1692, 4)
Cleaned Shape  (495, 4)
